In [2]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

In [3]:
import os
import sys
sys.path.append(os.path.abspath("../src"))

In [4]:
from config import *
from data_loader import MarketDataLoader

In [5]:
import importlib
import data_loader

importlib.reload(data_loader)

from data_loader import MarketDataLoader

import inspect

print(inspect.signature(MarketDataLoader.download_data))

(self, ticker, start_date='2015-01-01', end_date=None, force_download=False)


In [6]:
import yfinance as yf
print(yf.__version__)

1.5.1


In [7]:
loader = MarketDataLoader()
df = loader.download_data("AAPL")
print(df.shape)

df.head()

Loading cached data for AAPL
(2896, 8)


,Date,Adj Close,Close,High,Low,Open,Volume,Asset
0,2015-01-02,24.192606,27.332500,27.860001,26.837500,27.847500,212818400,AAPL
1,2015-01-05,23.511059,26.562500,27.162500,26.352501,27.072500,257142000,AAPL
2,2015-01-06,23.513268,26.565001,26.857500,26.157499,26.635000,263188400,AAPL
3,2015-01-07,23.842979,26.937500,27.049999,26.674999,26.799999,160423600,AAPL
4,2015-01-08,24.759087,27.972500,28.037500,27.174999,27.307501,237458000,AAPL


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2896 entries, 0 to 2895
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       2896 non-null   datetime64[us]
 1   Adj Close  2896 non-null   float64       
 2   Close      2896 non-null   float64       
 3   High       2896 non-null   float64       
 4   Low        2896 non-null   float64       
 5   Open       2896 non-null   float64       
 6   Volume     2896 non-null   int64         
 7   Asset      2896 non-null   str           
dtypes: datetime64[us](1), float64(5), int64(1), str(1)
memory usage: 192.4 KB


In [11]:
df.describe()

,Date,Adj Close,Close,High,Low,Open,Volume
count,2896,2896.000000,2896.000000,2896.000000,2896.000000,2896.000000,2.896000e+03
mean,2020-10-03 01:55:21.546961,113.709268,116.123166,117.288620,114.854695,116.021804,1.086653e+08
min,2015-01-02 00:00:00,20.565868,22.584999,22.917500,22.367500,22.500000,1.791060e+07
25%,2017-11-14 18:00:00,37.348959,39.970001,40.310000,39.665625,40.012501,6.093172e+07
50%,2020-10-01 12:00:00,112.303539,115.779999,117.384998,114.204998,116.022499,9.144640e+07
75%,2023-08-18 18:00:00,173.643097,176.130001,178.075001,174.485001,176.207497,1.346152e+08
max,2026-07-10 00:00:00,316.220001,316.220001,317.399994,312.170013,315.290009,6.488252e+08
std,NaN,80.612104,80.116352,80.916337,79.251504,80.037093,6.787495e+07


In [13]:
df.duplicated().sum()

np.int64(0)

In [14]:
from feature_engineering import FeatureEngineer

engineer = FeatureEngineer()

df = engineer.add_price_features(df)
print(df.shape)
print(df.columns)

(2896, 15)
Index(['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'Asset',
       'Daily_Return', 'Log_Return', 'Return_Lag_1', 'Return_Lag_5',
       'Rolling_STD_20', 'Rolling_STD_10', 'Intraday_Range'],
      dtype='str')


In [15]:
df = engineer.add_trend_features(df)
print(df.shape)

(2896, 23)


In [16]:
df = engineer.add_momentum_features(df)
print(df.shape)

(2896, 26)


In [17]:
df = engineer.add_volatility_features(df)
print(df.shape)

(2896, 31)


In [19]:
df = engineer.generate_features(df)
print(df.shape)

(2896, 31)


In [20]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2896 entries, 0 to 2895
Data columns (total 31 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Date                      2896 non-null   datetime64[us]
 1   Adj Close                 2896 non-null   float64       
 2   Close                     2896 non-null   float64       
 3   High                      2896 non-null   float64       
 4   Low                       2896 non-null   float64       
 5   Open                      2896 non-null   float64       
 6   Volume                    2896 non-null   int64         
 7   Asset                     2896 non-null   str           
 8   Daily_Return              2895 non-null   float64       
 9   Log_Return                2895 non-null   float64       
 10  Return_Lag_1              2894 non-null   float64       
 11  Return_Lag_5              2890 non-null   float64       
 12  Rolling_STD_20            2877 

In [21]:
feature_cols = [
    "Daily_Return", "Log_Return", "Return_Lag_1", "Return_Lag_5",
    "Intraday_Range", "Rolling_STD_10", "Rolling_STD_20",
    "SMA_10", "SMA_20", "EMA_10", "EMA_20",
    "EMA_12", "EMA_26", "MACD", "MACD_Signal",
    "RSI_14", "ROC_10", "Momentum_10",
    "BB_Middle", "BB_Upper", "BB_Lower",
    "ATR_14", "Historical_Volatility_20"
]

missing = [col for col in feature_cols if col not in df.columns]
print("Missing features:", missing)

Missing features: []


In [22]:
from target_engineering import TargetEngineer

target_engineer = TargetEngineer()

df = target_engineer.add_target_features(df)

In [24]:
df[["Target_Volatility_20"]].tail(25)

,Target_Volatility_20
2871,0.387112
2872,0.384992
2873,0.378697
2874,0.350495
2875,0.351453
2876,NaN
2877,NaN
2878,NaN
2879,NaN
2880,NaN


In [25]:
df["Target_Volatility_20"].describe()

count    2876.000000
mean        0.259865
std         0.123844
min         0.076442
25%         0.180359
50%         0.234980
75%         0.303886
max         1.079532
Name: Target_Volatility_20, dtype: float64

In [26]:
corr = (
    df.corr(numeric_only=True)["Target_Volatility_20"]
      .sort_values(ascending=False)
)

corr

Target_Volatility_20        1.000000
Historical_Volatility_20    0.450767
Intraday_Range              0.426863
Rolling_STD_20              0.258901
ATR_14                      0.252102
Rolling_STD_10              0.237910
Volume                      0.234027
BB_Upper                    0.058126
BB_Middle                   0.043117
SMA_20                      0.043117
EMA_26                      0.039820
EMA_20                      0.037699
SMA_10                      0.035525
EMA_12                      0.033527
EMA_10                      0.032156
High                        0.027023
BB_Lower                    0.026259
Open                        0.024249
Close                       0.023273
Adj Close                   0.022150
Low                         0.020687
Return_Lag_5               -0.073625
Return_Lag_1               -0.102434
Daily_Return               -0.105314
Log_Return                 -0.112641
MACD_Signal                -0.145892
MACD                       -0.190090
M

In [27]:
import os

os.makedirs("../data/processed", exist_ok=True)

df.to_csv("../data/processed/AAPL_engineered.csv", index=False)